Load data

In [7]:
# not good practice in production
import warnings
warnings.filterwarnings('ignore') 

In [8]:
import pandas as pd
from datasets import Dataset

data = pd.read_csv('https://raw.githubusercontent.com/bibasrairockz/Deployment/refs/heads/main/Sentiment_Classification/IMDB-Dataset.csv')
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
dataset = Dataset.from_pandas(data)
dataset = dataset.train_test_split(test_size=0.3)
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 35000
    })
    test: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 15000
    })
})

In [10]:
# input_ids, attention_mask, labels -> numbers

In [11]:
data['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [12]:
label2id = {'negative':0, 'positive':1}
id2label = {0:'negative', 1:'positive'}

dataset = dataset.map(lambda x: {'label': label2id[x['sentiment']]})

Map: 100%|██████████| 15000/15000 [00:02<00:00, 7425.48 examples/s]


In [13]:
dataset['train'][0]

{'review': 'This film uses all art-house clichés (slow pace, long static shots, minimal amount of dialog) to try to hide the fact that there really is nothing worth watching here: There is no plot to speak of, the characters are dreary (female lead) or cliché (Tersteeghe\'s character), and they do not ever talk to each other about anything that concerns their rather uneventful lives. The film is centered around a woman who finds out about her husbands adultery. Instead of confronting him, she half-heartedly takes revenge by committing adultery herself. After a fight and a reconciliation with her sister - who knew about the adultery without telling her - she asks her husband to stop cheating on her. They seem to be re-united as a family. Two other story lines - the planned move of the woman\'s elderly father with his young wife to Guernsey and the rivalry with the woman\'s sister - do not offer any interesting developments. The suicide of a colleague of the woman that seems to set off e

Data tokenization

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
from transformers import AutoTokenizer

model_ckpt = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt, use_fast=True)

In [15]:
tokenizer(dataset['train'][0]['review'])

{'input_ids': [101, 2023, 2143, 3594, 2035, 2396, 1011, 2160, 18856, 17322, 2015, 1006, 4030, 6393, 1010, 2146, 10763, 7171, 1010, 10124, 3815, 1997, 13764, 8649, 1007, 2000, 3046, 2000, 5342, 1996, 2755, 2008, 2045, 2428, 2003, 2498, 4276, 3666, 2182, 1024, 2045, 2003, 2053, 5436, 2000, 3713, 1997, 1010, 1996, 3494, 2024, 2852, 14644, 2100, 1006, 2931, 2599, 1007, 2030, 18856, 17322, 1006, 28774, 13473, 13910, 5369, 1005, 1055, 2839, 1007, 1010, 1998, 2027, 2079, 2025, 2412, 2831, 2000, 2169, 2060, 2055, 2505, 2008, 5936, 2037, 2738, 17837, 24475, 5313, 3268, 1012, 1996, 2143, 2003, 8857, 2105, 1037, 2450, 2040, 4858, 2041, 2055, 2014, 19089, 29169, 1012, 2612, 1997, 26964, 2032, 1010, 2016, 2431, 1011, 18627, 2135, 3138, 7195, 2011, 16873, 29169, 2841, 1012, 2044, 1037, 2954, 1998, 1037, 16088, 2007, 2014, 2905, 1011, 2040, 2354, 2055, 1996, 29169, 2302, 4129, 2014, 1011, 2016, 5176, 2014, 3129, 2000, 2644, 16789, 2006, 2014, 1012, 2027, 4025, 2000, 2022, 2128, 1011, 2142, 2004, 1037

In [17]:
def tokenize(batch):
    temp = tokenizer(batch['review'], padding=True, truncation=True, max_length=300)
    return temp

dataset = dataset.map(tokenize, batched=True, batch_size=None)

Map: 100%|██████████| 15000/15000 [00:11<00:00, 1283.48 examples/s]


In [30]:
dataset['train'][0].keys()

dict_keys(['review', 'sentiment', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

Model Evaluation function

In [43]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=1)

    return accuracy.compute(predictions=predictions, references=labels)

Model building and training

In [45]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=len(label2id), label2id=label2id, id2label=id2label)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 312, padding_idx=0)
      (position_embeddings): Embedding(512, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-1

In [47]:
args = TrainingArguments(
    output_dir='train_dir',
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

In [48]:
trainer.train()

 15%|█▌        | 500/3282 [04:21<24:11,  1.92it/s]

{'loss': 0.4596, 'grad_norm': 8.159070014953613, 'learning_rate': 1.695307739183425e-05, 'epoch': 0.46}


 30%|███       | 1000/3282 [12:27<1:03:17,  1.66s/it]

{'loss': 0.3537, 'grad_norm': 24.927953720092773, 'learning_rate': 1.3906154783668494e-05, 'epoch': 0.91}


                                                     
 33%|███▎      | 1094/3282 [16:35<31:33,  1.16it/s]

...

 91%|█████████▏| 3000/3282 [31:02<01:52,  2.50it/s]

{'loss': 0.2506, 'grad_norm': 11.885642051696777, 'learning_rate': 1.7184643510054846e-06, 'epoch': 2.74}


                                                   
100%|██████████| 3282/3282 [34:11<00:00,  1.60it/s]

{'eval_loss': 0.2961154282093048, 'eval_accuracy': 0.8783333333333333, 'eval_runtime': 74.5593, 'eval_samples_per_second': 201.182, 'eval_steps_per_second': 6.29, 'epoch': 3.0}
{'train_runtime': 2051.1664, 'train_samples_per_second': 51.19, 'train_steps_per_second': 1.6, 'train_loss': 0.3166984032741921, 'epoch': 3.0}


TrainOutput(global_step=3282, training_loss=0.3166984032741921, metrics={'train_runtime': 2051.1664, 'train_samples_per_second': 51.19, 'train_steps_per_second': 1.6, 'total_flos': 882184338000000.0, 'train_loss': 0.3166984032741921, 'epoch': 3.0})

In [49]:
trainer.evaluate()

100%|██████████| 469/469 [01:13<00:00,  6.39it/s]


{'eval_loss': 0.2961154282093048,
 'eval_accuracy': 0.8783333333333333,
 'eval_runtime': 73.6338,
 'eval_samples_per_second': 203.711,
 'eval_steps_per_second': 6.369,
 'epoch': 3.0}

Save and inference

In [50]:
trainer.save_model('tinybert-sentiment-analysis')

In [51]:
data = ['this movie was horrible, the plot was really boring. acting was okay',
        'the movie is really sucked. there is not plot and acting was bad',
        'what a beautiful movie. great plot. acting was good. will see it again']

In [55]:
from transformers import pipeline
import torch

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

classifier = pipeline('text-classification', model='tinybert-sentiment-analysis', device=device)

classifier(data)

[{'label': 'negative', 'score': 0.9900338053703308},
 {'label': 'negative', 'score': 0.9899500608444214},
 {'label': 'positive', 'score': 0.9903436899185181}]

Push model to S3

In [60]:
# create bucket
import boto3

s3 =  boto3.client('s3')

bucket_name = 'mlops-kgptalkie2'

def create_bucket(bucket_name):
    response = s3.list_buckets()
    buckets = [buck['Name'] for buck in response['Buckets']]
    if bucket_name not in buckets:
        s3.create_bucket(Bucket=bucket_name)
        print("Bucket is created")
    else:
        print("Bucket already exists")    

create_bucket(bucket_name)

Bucket already exists


In [ ]:
# Push model to s3 bucket ml-models/tinybert-sentiment-analysis
import os
import boto3

s3 = boto3.client('s3')
def upload_directory(directory_path, s3_prefix):
    for root, dir, files in os.walk(directory_path):
        for file in files:
            file_path = os.path.join(root, file).replace('\\', '/')
            relpath = os.path.relpath(file_path, directory_path)
            s3_key = os.path.join(s3_prefix, relpath).replace('\\','/')

            s3.upload_file(file_path, bucket_name,  s3_key)  
    print("Directory Uploaded")

upload_directory('tinybert-sentiment-analysis', 'ml-models/tinybert-sentiment-analysis')

In [ ]:
# s3://mlops-kgptalkie2/ml-models/tinybert-sentiment-analysis/